In [2]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_chimere_module import *
from expo_functions_module import *
from mortality_chimere_module import *
from association_module import *
from morbidity_chimere_module import *
print("Successfully loaded all modules")

loaded defined RR values
Successfully loaded all modules


In [3]:
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_sherpa(sc="s1", pol="ug_NO2", year=2019)

# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))

# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())

# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")

Concentrations in 2019 and 2019 are calculated for the pollutant 'ug_no2' (s1).
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [4]:
import os
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))
print(donnees_merged.head())

grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

     iriscod irisname comcod    comname depcod   depname  regcod    regname  \
0  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
1  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
2  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
3  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
4  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   

   age   pop2019   pop2030   pop2050  mort2019  mort2030  mort2050  \
0    0  1.001248  1.028860  1.007603  0.003590  0.003423  0.002972   
1    1  1.024924  1.022945  1.015467  0.000575  0.000556  0.000491   
2    2  1.049225  1.018921  1.023012  0.000261  0.000246  0.000219   
3    3  1.078030  1.016966  1.029914  0.000157  0.000155  0.000142   
4    4  1.114215  1.017214  1.035872  0.000121  0.000118  0.000109   

                                            geometry  
0  POLYGON ((1052346.7 6848413, 1052502.3 6848474

In [5]:
donnees_merged = donnees_merged[(donnees_merged["age"] >= 30) & (donnees_merged["age"] <= 99)]
cols_to_sum = ["pop2019", "pop2030", "pop2050", "mort2019", "mort2030", "mort2050"]
sum = donnees_merged[cols_to_sum].sum()
print(sum)

pop2019     4.138219e+07
pop2030     4.319732e+07
pop2050     4.516190e+07
mort2019    5.980088e+05
mort2030    6.938935e+05
mort2050    8.825646e+05
dtype: float64


In [8]:
import numpy as np
import pandas as pd


# ----------------------------
# Life Table Function
# ----------------------------
def life_expectancy_table(age, m_x, interval=1, a_x=0.5, radix=100000):
    n = len(age)

    if a_x is None or isinstance(a_x, (float, int)):
        a_x = np.full(n, a_x)

    m_x = np.nan_to_num(m_x)
    m_x[m_x < 0] = 0
    m_x = np.clip(m_x, 0, 1)

    q_x = (interval * m_x) / (1 + (interval - a_x) * m_x)
    q_x = np.minimum(q_x, 1.0)

    l_x = np.zeros(n)
    l_x[0] = radix
    for i in range(1, n):
        l_x[i] = l_x[i - 1] * (1 - q_x[i - 1])

    L_x = np.zeros(n)
    for i in range(n - 1):
        L_x[i] = interval * (l_x[i + 1] + a_x[i] * (l_x[i] - l_x[i + 1]))
    L_x[-1] = l_x[-1] / m_x[-1] if m_x[-1] > 0 else interval * l_x[-1]

    T_x = np.zeros(n)
    T_x[-1] = L_x[-1]
    for i in range(n - 2, -1, -1):
        T_x[i] = T_x[i + 1] + L_x[i]

    return T_x / np.where(l_x > 0, l_x, 1e-10)


# ----------------------------
# Inputs from donnees_merged
# ----------------------------
analysis_years = [2019, 2030, 2050]
ages = np.arange(30, 100)

donnees_merged_dynamic = donnees_merged.copy()
donnees_merged_dynamic["age"] = pd.to_numeric(donnees_merged_dynamic["age"], errors="coerce")
donnees_merged_dynamic = donnees_merged_dynamic[
    (donnees_merged_dynamic["age"] >= 30) & (donnees_merged_dynamic["age"] <= 99)
    ].copy()

for year in analysis_years:
    donnees_merged_dynamic[f"mort{year}"] = pd.to_numeric(
        donnees_merged_dynamic[f"mort{year}"], errors="coerce"
    ).fillna(0)
    donnees_merged_dynamic[f"pop{year}"] = pd.to_numeric(
        donnees_merged_dynamic[f"pop{year}"], errors="coerce"
    ).fillna(0)

data = {
    year: {
        "mort": donnees_merged_dynamic[f"mort{year}"].sum(),
        "pop": donnees_merged_dynamic[f"pop{year}"].sum(),
    }
    for year in analysis_years
}

pm25 = {2019: 8.80, 2030: 6.70, 2050: 5.31}
no2 = {2019: 11.19, 2030: 5.10, 2050: 2.49}

# ----------------------------
# WHO thresholds for 2019; relative thresholds for 2030 and 2050
# ----------------------------
pm25_thr = 5
no2_thr = 10

pm25_delta_scen = {
    2019: max(pm25[2019] - pm25_thr, 0),
    2030: max(pm25[2019] - pm25[2030], 0),
    2050: max(pm25[2019] - pm25[2050], 0),
}

no2_delta_scen = {
    2019: max(no2[2019] - no2_thr, 0),
    2030: max(no2[2019] - no2[2030], 0),
    2050: max(no2[2019] - no2[2050], 0),
}

# ----------------------------
# HRAPIE RR with CI
# ----------------------------
RR_PM25_Hrapie = 1.10  # exposure range 5-70
RR_PM25_Hrapie_high = 1.13
RR_PM25_Hrapie_low = 1.06
RR_NO2_Hrapie = 1.05  # exposure range 10-130
RR_NO2_Hrapie_high = 1.07
RR_NO2_Hrapie_low = 1.03

RR_PM25 = RR_PM25_Hrapie
RR_PM25_high = RR_PM25_Hrapie_high
RR_PM25_low = RR_PM25_Hrapie_low
RR_NO2 = RR_NO2_Hrapie
RR_NO2_high = RR_NO2_Hrapie_high
RR_NO2_low = RR_NO2_Hrapie_low

beta_pm25 = np.log(RR_PM25) / 10
beta_pm25_high = np.log(RR_PM25_high) / 10
beta_pm25_low = np.log(RR_PM25_low) / 10

beta_no2 = np.log(RR_NO2) / 10
beta_no2_high = np.log(RR_NO2_high) / 10
beta_no2_low = np.log(RR_NO2_low) / 10


# ----------------------------
# Helpers
# ----------------------------
def build_mx(total_deaths, total_pop):
    ages = np.arange(30, 100)
    # exponential mortality curve (Gompertz-like)
    base = 0.0005
    growth = 0.085
    m_x = base * np.exp(growth * (ages - 30))
    # scale to match total deaths
    scale = (total_deaths / total_pop) / np.mean(m_x)
    return m_x * scale


def build_dynamic_mx(year):
    return build_mx(
        data[year]["mort"],
        data[year]["pop"]
    )


def lnRR_and_sd(rr, lci, uci):
    lnrr = np.log(rr)
    lnlci = np.log(lci)
    lnuci = np.log(uci)
    sd = (lnuci - lnlci) / (2 * 1.96)
    return lnrr, sd


lnRR_pm25, sd_pm25 = lnRR_and_sd(RR_PM25, RR_PM25_low, RR_PM25_high)
lnRR_no2, sd_no2 = lnRR_and_sd(RR_NO2, RR_NO2_low, RR_NO2_high)

N_MC = 10000


def compute_LE_scenario(base_year, deltaC, beta):
    m_x = build_dynamic_mx(base_year)

    AF = 1 - np.exp(-beta * deltaC)
    m_x_corr = m_x * (1 - AF)

    le_base = life_expectancy_table(ages, m_x)
    le_corr = life_expectancy_table(ages, m_x_corr)

    return {
        "e30_gain_mo": (le_corr[0] - le_base[0]) * 12,
        "avg_LE_gain_mo": (np.mean(le_corr) - np.mean(le_base)) * 12
    }


def MC_LE_scenario(base_year, deltaC, lnRR_mean, lnRR_sd):
    m_x = build_dynamic_mx(base_year)
    beta_draws = np.random.normal(loc=lnRR_mean / 10, scale=lnRR_sd / 10, size=N_MC)
    AF_draws = 1 - np.exp(-beta_draws * deltaC)

    le_base = life_expectancy_table(ages, m_x)

    e30_gains = []
    avgLE_gains = []
    for AF in AF_draws:
        m_x_corr = m_x * (1 - AF)
        le_corr = life_expectancy_table(ages, m_x_corr)
        e30_gains.append((le_corr[0] - le_base[0]) * 12)
        avgLE_gains.append((np.mean(le_corr) - np.mean(le_base)) * 12)

    return np.array(e30_gains), np.array(avgLE_gains)


# ----------------------------
# RUN
# ----------------------------
results = []

for year in [2019, 2030, 2050]:
    e30_pm25, avg_pm25 = MC_LE_scenario(
        year, pm25_delta_scen[year], lnRR_pm25, sd_pm25
    )

    e30_no2, avg_no2 = MC_LE_scenario(
        year, no2_delta_scen[year], lnRR_no2, sd_no2
    )

    results.append({
        "Scenario": "2019 → WHO" if year == 2019 else f"2019 → {year}",

        "Mortality_year": year,
        "Total_deaths": data[year]["mort"],
        "Total_population": data[year]["pop"],

        "PM25_delta": pm25_delta_scen[year],
        "PM25_e30_gain_months": np.median(e30_pm25),
        "PM25_e30_gain_months_LCI": np.percentile(e30_pm25, 2.5),
        "PM25_e30_gain_months_UCI": np.percentile(e30_pm25, 97.5),
        "PM25_avg_LE_gain_months": np.median(avg_pm25),
        "PM25_avg_LE_gain_months_LCI": np.percentile(avg_pm25, 2.5),
        "PM25_avg_LE_gain_months_UCI": np.percentile(avg_pm25, 97.5),

        "NO2_delta": no2_delta_scen[year],
        "NO2_e30_gain_months": np.median(e30_no2),
        "NO2_e30_gain_months_LCI": np.percentile(e30_no2, 2.5),
        "NO2_e30_gain_months_UCI": np.percentile(e30_no2, 97.5),
        "NO2_avg_LE_gain_months": np.median(avg_no2),
        "NO2_avg_LE_gain_months_LCI": np.percentile(avg_no2, 2.5),
        "NO2_avg_LE_gain_months_UCI": np.percentile(avg_no2, 97.5),
    })

df_results = pd.DataFrame(results)

pd.set_option("display.float_format", "{:.2f}".format)
print(df_results)


      Scenario  Mortality_year  Total_deaths  Total_population  PM25_delta  \
0   2019 → WHO            2019     598008.82       41382192.15        3.80   
1  2019 → 2030            2030     693893.53       43197315.06        2.10   
2  2019 → 2050            2050     882564.58       45161902.59        3.49   

   PM25_e30_gain_months  PM25_e30_gain_months_LCI  PM25_e30_gain_months_UCI  \
0                  7.10                      4.69                      9.52   
1                  3.70                      2.44                      4.95   
2                  5.63                      3.74                      7.50   

   PM25_avg_LE_gain_months  PM25_avg_LE_gain_months_LCI  \
0                     6.40                         4.22   
1                     3.27                         2.16   
2                     4.82                         3.21   

   PM25_avg_LE_gain_months_UCI  NO2_delta  NO2_e30_gain_months  \
0                         8.59       1.19                 1.13   
1

In [7]:
import numpy as np
import pandas as pd

# ------------------------------------
# Life Table Function
# ------------------------------------
def life_expectancy_table(age, m_x, interval=1, a_x=0.5, radix=100000):
    age = np.asarray(age)
    m_x = np.asarray(m_x, dtype=float)
    n = len(age)

    if len(m_x) != n:
        raise ValueError(f"`m_x` length ({len(m_x)}) must match `age` length ({n}).")

    if a_x is None:
        a_x = np.full(n, 0.5, dtype=float)
    elif np.isscalar(a_x):
        a_x = np.full(n, float(a_x), dtype=float)
    else:
        a_x = np.asarray(a_x, dtype=float)
        if len(a_x) != n:
            raise ValueError(f"`a_x` length ({len(a_x)}) must match `age` length ({n}).")

    m_x = np.nan_to_num(m_x, nan=0.0, posinf=1.0, neginf=0.0)
    m_x[m_x < 0] = 0
    m_x = np.clip(m_x, 0, 1)

    q_x = (interval * m_x) / (1 + (interval - a_x) * m_x)
    q_x = np.minimum(q_x, 1.0)

    l_x = np.zeros(n)
    l_x[0] = radix
    for i in range(1, n):
        l_x[i] = l_x[i - 1] * (1 - q_x[i - 1])

    L_x = np.zeros(n)
    for i in range(n - 1):
        L_x[i] = interval * (l_x[i + 1] + a_x[i] * (l_x[i] - l_x[i + 1]))
    L_x[-1] = l_x[-1] / m_x[-1] if m_x[-1] > 0 else interval * l_x[-1]

    T_x = np.zeros(n)
    T_x[-1] = L_x[-1]
    for i in range(n - 2, -1, -1):
        T_x[i] = T_x[i + 1] + L_x[i]

    return T_x / np.where(l_x > 0, l_x, 1e-10)


# ------------------------------------
# Inputs from donnees_merged
# ------------------------------------
analysis_years = [2019, 2030, 2050]
ages = np.arange(30, 100)

donnees_merged_dynamic = donnees_merged.copy()
donnees_merged_dynamic["age"] = pd.to_numeric(donnees_merged_dynamic["age"], errors="coerce")
donnees_merged_dynamic = donnees_merged_dynamic[
    (donnees_merged_dynamic["age"] >= 30) & (donnees_merged_dynamic["age"] <= 99)
    ].copy()

for year in analysis_years:
    donnees_merged_dynamic[f"mort{year}"] = pd.to_numeric(
        donnees_merged_dynamic[f"mort{year}"], errors="coerce"
    ).fillna(0)
    donnees_merged_dynamic[f"pop{year}"] = pd.to_numeric(
        donnees_merged_dynamic[f"pop{year}"], errors="coerce"
    ).fillna(0)

# Use observed national age-specific mortality rates per year
m_x_dict = {}
for year in analysis_years:
    age_grouped = (
        donnees_merged_dynamic
        .groupby("age", as_index=False)[[f"mort{year}", f"pop{year}"]]
        .sum()
        .set_index("age")
        .reindex(ages, fill_value=0)
    )

    deaths = age_grouped[f"mort{year}"].to_numpy(dtype=float)
    pop = age_grouped[f"pop{year}"].to_numpy(dtype=float)

    m_x = np.divide(deaths, pop, out=np.zeros_like(deaths, dtype=float), where=pop > 0)
    m_x_dict[year] = m_x

# ------------------------------------
# Exposure levels and thresholds
# ------------------------------------
pm25 = {2019: 8.80, 2030: 6.70, 2050: 5.31}
no2 = {2019: 11.19, 2030: 5.10, 2050: 2.49}
pm25_thr = 5
no2_thr = 10

pm25_delta_scen = {
    2019: max(pm25[2019] - pm25_thr, 0),
    2030: max(pm25[2019] - pm25[2030], 0),
    2050: max(pm25[2019] - pm25[2050], 0),
}
no2_delta_scen = {
    2019: max(no2[2019] - no2_thr, 0),
    2030: max(no2[2019] - no2[2030], 0),
    2050: max(no2[2019] - no2[2050], 0),
}


# ------------------------------------
# HRAPIE RR, derive lnRR mean and SD (MC simulation)
# ------------------------------------
def lnRR_and_sd(rr, lci, uci):
    lnrr = np.log(rr)
    lnlci = np.log(lci)
    lnuci = np.log(uci)
    sd = (lnuci - lnlci) / (2 * 1.96)
    return lnrr, sd


# PM2.5, NO2 RR vals from HRAPIE
RR_PM25, RR_PM25_low, RR_PM25_high = 1.10, 1.06, 1.13
RR_NO2, RR_NO2_low, RR_NO2_high = 1.05, 1.03, 1.07

lnRR_pm25, sd_pm25 = lnRR_and_sd(RR_PM25, RR_PM25_low, RR_PM25_high)
lnRR_no2, sd_no2 = lnRR_and_sd(RR_NO2, RR_NO2_low, RR_NO2_high)

# ------------------------------------
# MC simulation parameters
# ------------------------------------
N_MC = 10000  # number of Monte Carlo draws


# ------------------------------------
# MAIN MC FUNCTION
# ------------------------------------
def MC_LE_gain(mx_base, deltaC, lnRR_mean, lnRR_sd):
    mx_base = np.asarray(mx_base, dtype=float)
    beta_draws = np.random.normal(loc=lnRR_mean / 10, scale=lnRR_sd / 10, size=N_MC)
    AF_draws = 1 - np.exp(-beta_draws * deltaC)

    le_base = life_expectancy_table(ages, mx_base)

    e30_gains = []
    avgLE_gains = []
    for AF in AF_draws:
        mx_new = mx_base * (1 - AF)
        le_new = life_expectancy_table(ages, mx_new)
        e30_gains.append((le_new[0] - le_base[0]) * 12)
        avgLE_gains.append((np.mean(le_new) - np.mean(le_base)) * 12)

    return np.array(e30_gains), np.array(avgLE_gains)


# ------------------------------------
# RUN MC FOR ALL SCENARIOS
# ------------------------------------
results = []
for year in analysis_years:
    # PM2.5 simulation
    e30_pm25, avg_pm25 = MC_LE_gain(m_x_dict[year], pm25_delta_scen[year], lnRR_pm25, sd_pm25)
    # NO2 simulation
    e30_no2, avg_no2 = MC_LE_gain(m_x_dict[year], no2_delta_scen[year], lnRR_no2, sd_no2)

    results.append({
        "Scenario": "2019 → WHO" if year == 2019 else f"2019 → {year}",
        "Mortality_year": year,
        "PM25_delta": pm25_delta_scen[year],
        "PM25_e30_gain_months": np.median(e30_pm25),
        "PM25_e30_gain_months_LCI": np.percentile(e30_pm25, 2.5),
        "PM25_e30_gain_months_UCI": np.percentile(e30_pm25, 97.5),
        "PM25_avg_LE_gain_months": np.median(avg_pm25),
        "PM25_avg_LE_gain_months_LCI": np.percentile(avg_pm25, 2.5),
        "PM25_avg_LE_gain_months_UCI": np.percentile(avg_pm25, 97.5),

        "NO2_delta": no2_delta_scen[year],
        "NO2_e30_gain_months": np.median(e30_no2),
        "NO2_e30_gain_months_LCI": np.percentile(e30_no2, 2.5),
        "NO2_e30_gain_months_UCI": np.percentile(e30_no2, 97.5),
        "NO2_avg_LE_gain_months": np.median(avg_no2),
        "NO2_avg_LE_gain_months_LCI": np.percentile(avg_no2, 2.5),
        "NO2_avg_LE_gain_months_UCI": np.percentile(avg_no2, 97.5),
    })

df_results = pd.DataFrame(results)
pd.set_option("display.float_format", "{:.2f}".format)
print(df_results)


      Scenario  Mortality_year  PM25_delta  PM25_e30_gain_months  \
0   2019 → WHO            2019        3.80                  4.03   
1  2019 → 2030            2030        2.10                  2.12   
2  2019 → 2050            2050        3.49                  3.13   

   PM25_e30_gain_months_LCI  PM25_e30_gain_months_UCI  \
0                      2.69                      5.35   
1                      1.41                      2.83   
2                      2.08                      4.16   

   PM25_avg_LE_gain_months  PM25_avg_LE_gain_months_LCI  \
0                     2.75                         1.83   
1                     1.48                         0.98   
2                     2.24                         1.49   

   PM25_avg_LE_gain_months_UCI  NO2_delta  NO2_e30_gain_months  \
0                         3.65       1.19                 0.65   
1                         1.97       6.09                 3.16   
2                         2.99       8.70                 4.00 

In [10]:
# National avoided mortality using the same AF formula and concentration reductions as LE analysis
national_data = {
    2019: {"pop": 4.138219e07, "mort": 5.980088e05},
    2030: {"pop": 4.319732e07, "mort": 6.938935e05},
    2050: {"pop": 4.516190e07, "mort": 8.825646e05},
}

def avoided_mortality_national(total_mortality, delta_c, rr):
    af = 1 - np.exp(-np.log(rr) * delta_c / 10)
    return total_mortality * af, af


national_avoided_results = []

for year in analysis_years:
    pm25_avoided, pm25_af = avoided_mortality_national(
        national_data[year]["mort"],
        pm25_delta_scen[year],
        RR_PM25
    )
    pm25_avoided_lci, pm25_af_lci = avoided_mortality_national(
        national_data[year]["mort"],
        pm25_delta_scen[year],
        RR_PM25_low
    )
    pm25_avoided_uci, pm25_af_uci = avoided_mortality_national(
        national_data[year]["mort"],
        pm25_delta_scen[year],
        RR_PM25_high
    )

    no2_avoided, no2_af = avoided_mortality_national(
        national_data[year]["mort"],
        no2_delta_scen[year],
        RR_NO2
    )
    no2_avoided_lci, no2_af_lci = avoided_mortality_national(
        national_data[year]["mort"],
        no2_delta_scen[year],
        RR_NO2_low
    )
    no2_avoided_uci, no2_af_uci = avoided_mortality_national(
        national_data[year]["mort"],
        no2_delta_scen[year],
        RR_NO2_high
    )

    national_avoided_results.append({
        "Scenario": "2019 → WHO" if year == 2019 else f"2019 → {year}",
        "Mortality_year": year,
        "Total_population": national_data[year]["pop"],
        "Total_deaths": national_data[year]["mort"],

        "PM25_delta": pm25_delta_scen[year],
        "PM25_AF": pm25_af,
        "PM25_AF_LCI": pm25_af_lci,
        "PM25_AF_UCI": pm25_af_uci,
        "PM25_avoided_mortality": pm25_avoided,
        "PM25_avoided_mortality_LCI": pm25_avoided_lci,
        "PM25_avoided_mortality_UCI": pm25_avoided_uci,

        "NO2_delta": no2_delta_scen[year],
        "NO2_AF": no2_af,
        "NO2_AF_LCI": no2_af_lci,
        "NO2_AF_UCI": no2_af_uci,
        "NO2_avoided_mortality": no2_avoided,
        "NO2_avoided_mortality_LCI": no2_avoided_lci,
        "NO2_avoided_mortality_UCI": no2_avoided_uci,
    })

df_national_avoided_mortality = pd.DataFrame(national_avoided_results)

pd.set_option("display.float_format", "{:.2f}".format)
print(df_national_avoided_mortality)

      Scenario  Mortality_year  Total_population  Total_deaths  PM25_delta  \
0   2019 → WHO            2019       41382190.00     598008.80        3.80   
1  2019 → 2030            2030       43197320.00     693893.50        2.10   
2  2019 → 2050            2050       45161900.00     882564.60        3.49   

   PM25_AF  PM25_AF_LCI  PM25_AF_UCI  PM25_avoided_mortality  \
0     0.04         0.02         0.05                21271.08   
1     0.02         0.01         0.03                13750.31   
2     0.03         0.02         0.04                28874.08   

   PM25_avoided_mortality_LCI  PM25_avoided_mortality_UCI  NO2_delta  NO2_AF  \
0                    13095.70                    27138.08       1.19    0.01   
1                     8439.07                    17582.66       6.09    0.03   
2                    17766.44                    36853.31       8.70    0.04   

   NO2_AF_LCI  NO2_AF_UCI  NO2_avoided_mortality  NO2_avoided_mortality_LCI  \
0        0.00        0.01     